In [1]:
!git clone https://github.com/flatironinstitute/ironclust

fatal: destination path 'ironclust' already exists and is not an empty directory.


In [ ]:
import spikeinterface as si 
import spikeinterface.sorters as ss
from tempfile import TemporaryDirectory
import numpy as np
import spikeinterface as si
import spikeinterface.preprocessing as spre
import os

In [3]:
ss.get_sorter_params_description("ironclust")

{'detect_sign': 'Use -1 (negative), 1 (positive) or 0 (both) depending on the sign of the spikes in the recording',
 'adjacency_radius': 'Use -1 to include all channels in every neighborhood',
 'adjacency_radius_out': 'Use -1 to include all channels in every neighborhood',
 'detect_threshold': 'detection threshold',
 'prm_template_name': '.prm template file name',
 'freq_min': 'High-pass filter cutoff frequency',
 'freq_max': 'Low-pass filter cutoff frequency',
 'merge_thresh': 'Threshold for automated merging',
 'pc_per_chan': 'Number of principal components per channel',
 'whiten': 'Whether to do channel whitening as part of preprocessing',
 'filter_type': 'Filter type: none, bandpass, wiener, fftdiff, ndiff',
 'filter_detect_type': 'Filter type for detection: none, bandpass, wiener, fftdiff, ndiff',
 'common_ref_type': 'Common reference type: none, mean, median, trimmean',
 'batch_sec_drift': 'Batch duration in seconds. clustering time duration',
 'step_sec_drift': 'Compute anatomic

In [4]:
!which matlab

which: no matlab in (/mpcdf/soft/SLE_15/packages/x86_64/anaconda/3/2023.03/bin:/raven/u/gongt/.vscode-server/cli/servers/Stable-bf9252a2fb45be6893dd8870c0bf37e2e1766d61/server/bin/remote-cli:/mpcdf/soft/SLE_15/packages/x86_64/anaconda/3/2023.03/bin:/mpcdf/soft/SLE_15/packages/x86_64/anaconda/3/2023.03/condabin:/mpcdf/soft/SLE_15/packages/x86_64/datashare/0.4.1/exe:/mpcdf/soft/SLE_15/packages/x86_64/anaconda/3/2023.03/bin:/mpcdf/soft/SLE_15/packages/skylake/fftw/intel_21.4.0-2021.4.0-impi_2021.4-2021.4.0/3.3.10/bin:/mpcdf/soft/SLE_15/packages/x86_64/intel_oneapi/2021.4/mpi/2021.4.0/bin:/mpcdf/soft/SLE_15/packages/x86_64/intel/2021.4.0/bin:/mpcdf/soft/SLE_15/packages/x86_64/intel_oneapi/2021.4/compiler/latest/linux/bin/intel64:/mpcdf/soft/SLE_15/packages/x86_64/intel_oneapi/2021.4/compiler/latest/linux/bin:/mpcdf/soft/SLE_15/packages/x86_64/Modules/5.4.0/bin:/u/gongt/.local/bin:/u/gongt/bin:/usr/local/bin:/usr/bin:/bin:/usr/lib/mit/bin:/usr/lib/mit/sbin:/afs/ipp/amd64_sles15/bin:/mpcdf/s

In [21]:
root_folder = '/nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_g_g0'
data_folder = f'{root_folder}/preprocessed_slice_550_1270s'
exp_folder = os.path.join(root_folder, 'IC_test_550_1270s')
rec = si.load_extractor(data_folder)
rec

/tmp/ipykernel_88932/2775694490.py:4: DeprecationWarning: load_extractor() is deprecated and will be removed in version 0.104.0. Please use load() instead.
  rec = si.load_extractor(data_folder)


BinaryFolderRecording: 355 channels - 30000.283521 Hz - 1 segments - 21,600,205 samples 
                       720.00s (12.00 minutes) - float16 dtype - 14.28 GiB

In [22]:
rec = spre.astype(rec, dtype="int16", round = False)

In [ ]:
ss.IronClustSorter.set_ironclust_path("/u/gongt/scripts/ironclust")
sorting = ss.run_sorter(sorter_name="ironclust", recording=rec, folder=exp_folder)

Setting IRONCLUST_PATH environment variable for subprocess calls to: /u/gongt/scripts/ironclust


SpikeSortingError: Spike sorting error trace:
Traceback (most recent call last):
  File "/u/gongt/.local/lib/python3.10/site-packages/spikeinterface/sorters/basesorter.py", line 270, in run_from_folder
    SorterClass._run_from_folder(sorter_output_folder, sorter_params, verbose)
  File "/u/gongt/.local/lib/python3.10/site-packages/spikeinterface/sorters/external/ironclust.py", line 268, in _run_from_folder
    raise Exception("ironclust returned a non-zero exit code")
Exception: ironclust returned a non-zero exit code

Spike sorting failed. You can inspect the runtime trace in /nexus/posix0/MBR-neuralsystems/ephys/GLC6962/GLC6962_g_g0/IC_test_550_1270s/spikeinterface_log.json.

In [ ]:
def smart_compute(analyzer_save, sorting):
    if os.path.exists(analyzer_save):
        print("analyzer folder exists, loading extensions...")
        sorting_analyzer = si.load_sorting_analyzer(folder = analyzer_save, format = 'binary_folder')
    else: 
        print("analyzer folder does not exist, creating new analyzer...")
        sorting_analyzer = si.create_sorting_analyzer(sorting = sorting, recording = rec, format = 'binary_folder', folder = analyzer_save)
    
    existing_exts = sorting_analyzer.get_loaded_extension_names()
    print("already available:", existing_exts)

    #print("computing sparsity...")
    #sparsity = si.compute_sparsity(sorting_analyzer, method = 'closest_channels', num_channels = 48)
    #print("finished computing sparsity")

    compute_dict = {
    'random_spikes': {}, #random spike selection across units 
    'waveforms':{},
    'principal_components':{'n_components': 6, 'whiten':False},
    'templates':{
    }
    }

    to_compute = {
        ext: params for ext, params in compute_dict.items() if ext not in existing_exts
    }
    print("need to compute:", list(to_compute.keys()))

    if len(to_compute) > 0: 
        print("computing...")
        sorting_analyzer.compute(to_compute, save = True, progress_bar = True, n_jobs = 8)
    
    return sorting_analyzer 

In [ ]:
print(sorting)

In [ ]:
analyzer_save = os.path.join(os.path.join(root_folder,'ms5_test'), 'si_analyzer')
sorting_analyzer = smart_compute(analyzer_save,  sorting) 

In [ ]:
sorting_analyzer.compute('templates', save = True, progress_bar = True, n_jobs = 8)

In [ ]:
from spikeinterface.exporters import export_to_phy
export_to_phy(sorting_analyzer=sorting_analyzer, 
              output_folder= os.path.join(exp_folder, 'export_to_phy'),
              copy_binary = False, 
              remove_if_exists = True,
              n_jobs = 8
              )